# AI Tinkerers Milan: Multimodal Transformer for Structure Elucidation


In [ ]:
## 1. Problem Setting: Spectroscopy → Structure Elucidation (~3 min)
- Show 2 spectra: SMILES vs 3D Rendering  
- Briefly explain the goal: predicting molecular structure from spectral data.


## 2. Objective: End-to-End Multimodal Transformer (~1 min)
- Introduce the model architecture.  
- Goal: Predict SMILES string directly from multimodal spectral inputs (NMR, IR, Mass Spec).


## 3. Problem: Autoregressive Decoding Challenges (~3 min)
- Standard AR Transformers for sequence generation.  
- Introduce Beam Search: common approach.  
- Highlight Output Space Fragility: slight changes in input/decoding can lead to very different (often invalid) outputs.


## 4. Regular Test-Time Compute → Impossible! (~1 min)
- Decoding token-by-token is slow.  
- Generating diverse candidates (like in Beam Search) multiplies the cost.  
- Becomes computationally infeasible for large output spaces or real-time needs.


## Setup: Imports, Config, Model Loading


import torch
import numpy as np
from typing import List, Optional, Dict, Union, Tuple, Any
from enum import Enum
import heapq
import yaml
from pathlib import Path
import json
import os
import sys
from IPython.display import display, Markdown

# Add project root to sys.path to import custom modules
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))  
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from models.multimodal_to_smiles import MultiModalToSMILESModel
from models.smiles_tokenizer import SmilesTokenizer
from inference.inference import ModelInference, DecodingStrategy, BeamSearchNode, EntropixNode


In [ ]:
# --- Configuration Loading ---
def load_config(config_path=None):
    default_config = {
        'model': {
            'max_seq_length': 512,
            'max_nmr_length': 128,
            'max_memory_length': 128,
            'embed_dim': 256,
            'num_heads': 8,
            'num_layers': 6,
            'dropout': 0.1,
            'use_stablemax': False,
            'width_basis': 13
        },
        'data': {
            'tokenized_dir': '../tokenized_baseline/data'
        }
    }
    if config_path and os.path.exists(config_path):
        with open(config_path) as f:
            custom = yaml.safe_load(f)
        def update(d, u):
            for k,v in u.items():
                if isinstance(v, dict):
                    d[k] = update(d.get(k, {}), v)
                else:
                    d[k] = v
            return d
        update(default_config, custom)
    else:
        print(f"Warning: Config file '{config_path}' not found. Using default config.")
    return default_config

# --- Parameters (EDIT PATHS!) ---
CHECKPOINT_PATH = '../checkpoints/model_epoch_best.pth'
CONFIG_PATH     = '../configs/baseline_config.yaml'
SMILES_VOCAB    = '../inference/vocab.txt'
NMR_VOCAB_DIR   = '../tokenized_baseline/'

config = load_config(CONFIG_PATH)

# --- Device Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Tokenizers ---
try:
    smiles_tokenizer = SmilesTokenizer(vocab_file=SMILES_VOCAB)
    print(f"SMILES Tokenizer loaded from {SMILES_VOCAB}")
except FileNotFoundError:
    print(f"Error: SMILES vocab not found at {SMILES_VOCAB}")
    smiles_tokenizer = None

nmr_vocab_path = Path(NMR_VOCAB_DIR) / 'vocab.json'
try:
    with open(nmr_vocab_path) as f:
        nmr_tokenizer_map = json.load(f)
    print(f"NMR Tokenizer map loaded from {nmr_vocab_path}")
except FileNotFoundError:
    print(f"Error: NMR vocab not found at {nmr_vocab_path}")
    nmr_tokenizer_map = {}

# --- Model Init & Checkpoint Load ---
model = None
if smiles_tokenizer:
    smiles_size = len(smiles_tokenizer)
    nmr_size = max(nmr_tokenizer_map.values())+1 if nmr_tokenizer_map else 1
    model = MultiModalToSMILESModel(
        smiles_vocab_size=smiles_size,
        nmr_vocab_size=nmr_size,
        max_seq_length=config['model']['max_seq_length'],
        max_nmr_length=config['model']['max_nmr_length'],
        max_memory_length=config['model']['max_memory_length'],
        embed_dim=config['model']['embed_dim'],
        num_heads=config['model']['num_heads'],
        num_layers=config['model']['num_layers'],
        dropout=config['model']['dropout'],
        verbose=False,
        use_stablemax=config['model'].get('use_stablemax', False)
    ).to(device)
    print(f"Model initialized (SMILES vocab={smiles_size}, NMR vocab={nmr_size})")

    if os.path.exists(CHECKPOINT_PATH):
        ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
        try:
            model.load_state_dict(ckpt['model_state_dict'])
            print("Loaded weights from 'model_state_dict'")
        except KeyError:
            try:
                model.load_state_dict(ckpt)
                print("Loaded weights directly from checkpoint")
            except Exception as e:
                print(f"Failed to load weights: {e}")
                model = None
        if model: model.eval()
    else:
        print(f"Checkpoint not found at {CHECKPOINT_PATH}")
        model = None
else:
    print("Skipping model init due to tokenizer failure")

# --- Inference Wrapper ---
inference = None
if model and smiles_tokenizer:
    inference = ModelInference(model, smiles_tokenizer, device)
    print("ModelInference initialized")
else:
    print("Skipping inference init")


## 5. Latent Test-Time Compute (~5 min)
- **Explain Pretraining**: How the model learns representations.  
- **Explain Inference**: The standard autoregressive process.  
- **Why it’s not just a bigger model**: Needing *more compute at test time* for hard cases (an “accordion” architecture).


## Example Data Loading (Placeholder)
*You will need to replace this with your actual data loading logic.*


In [ ]:
# Placeholder for loading/defining example NMR, IR, Mass Spec data

# Example using dummy data (replace with actual loading and preprocessing)
example_nmr_tokens = None
# e.g. torch.randint(0, nmr_vocab_size, (1, config['model']['max_nmr_length'])).to(device)

example_ir_data = None
# e.g. torch.randn(1, config['model']['max_memory_length']).to(device)

example_mass_data = None
# e.g. torch.randn(1, some_mass_spec_feature_dim).to(device)

print(f"Example NMR tokens shape: {example_nmr_tokens.shape if example_nmr_tokens is not None else 'None'}")
print(f"Example IR data shape: {example_ir_data.shape if example_ir_data is not None else 'None'}")
print(f"Example Mass data shape: {example_mass_data.shape if example_mass_data is not None else 'None'}")
